# Batch: Évaluation de granularité dans des noyaux de levures

## 0. Imports requis et paramètres globaux

In [ ]:
from napari_yeasts_granularity.operators.data_loader import DataLoader
from napari_yeasts_granularity import FindCellsOperator
from napari_yeasts_granularity import (
    MeasureIntensitiesOperator,
    MeasureShapeOperator,
    MeasureSpotsOperator,
    MeasurementsManager,
    MeasureColocOperator
)
import tifffile as tiff
import xarray as xr
from pathlib import Path

- `channel_indices`: The rank of each channel in the input images.
    - "brightfield": The transmitted light channel.
    - "main": The channel used to segment nuclei and evaluate the granularity.
    - "secondary": The channel used to evaluate the colocalization with the 'main' channel.
- `images_root`: The root folder containing the hierarchy of images.
- `labels_root`: Where the label maps will be saved.
- `results_root`: Where the CSVs will be saved.
- `base_calibration`: The physical size of voxels for each axis (and 1.0 for time).

In [ ]:
channel_indices = {
    'brightfield': 0,
    'main'       : 1,
    'secondary'  : 2
}
images_root = Path("/home/clement/Documents/projects/2292-yeasts-granularity/2026-05-11-tiff")
labels_root = Path("/media/clement/fae534f3-f6ab-41aa-9554-baf3f7b791731/yeasts-granularity")
results_root = Path("/media/clement/fae534f3-f6ab-41aa-9554-baf3f7b791731/results")
base_calibration = {
    'T': 1.0,
    'Z': 0.3,
    'Y': 0.065,
    'X': 0.065
}

## 1. Segmentation des noyaux

### A. Paramètres de segmentation des noyaux

- `kill_borders`: Est-ce qu'on retire les noyaux qui touchent la bordure X, Y et Z ?
- `gaussian_sigma`: Sigma (≈ radius) du flou utilisé lors de la segmentation pour débruiter.
- `use_log`: Est-ce que la fonction logarithme est appliquée à l'image pour essayer d'attraper les noyaux les plus sombres ?
- `log_factor`: Facteur appliqué à l'image avant d'être passée au logarithme. Plus grand == noyaux plus sombres attrapés.
- `min_obj_size`: Taille minimale (en nombre de voxels) qu'un objet doit faire pour ne pas être considéré comme un débris.
- `objects_diam`: En µm, diamètre approximatif d'un noyau. Utile pour le tracking. Si un noyau bouge plus que cette distance, il est considéré perdu.

In [ ]:
kill_borders   = True # FindCellsOperator.default_kill_borders()
gaussian_sigma = 1.0  # FindCellsOperator.default_gaussian_sigma()
use_log        = True # FindCellsOperator.default_use_log()
log_factor     = 25.0 # FindCellsOperator.default_log_factor()
min_obj_size   = 100  # FindCellsOperator.default_min_obj_size()
objects_diam   = 2.7  # FindCellsOperator.default_objects_diam()

### B. Segmentation des noyaux

In [ ]:
dl = DataLoader(
    group_size=len(channel_indices), 
    group_index=channel_indices['main']
)
dl.set_root_paths(
    images_root, 
    labels_root, 
    results_root
)
dl.set_base_calibration(base_calibration)

while dl.next():
    image, image_axes = dl.load_image()
    calib = dl.get_calibration(image_axes)

    op = FindCellsOperator()
    op.set_input_image(image, axes=image_axes)
    op.set_calibration(calib)
    op.set_kill_borders(kill_borders)
    op.set_gaussian_sigma(gaussian_sigma)
    op.set_use_log(use_log)
    op.set_log_factor(log_factor)
    op.set_min_obj_size(min_obj_size)
    op.set_objects_diam(objects_diam)
    op.run()

    dl.save_labels(op.get_output_nuclei().values)

## 2. Mesures dans les noyaux

### A. Paramètres de mesure

In [ ]:
use_intensities_measurements = True
use_shape_measurements       = True
use_spots_count_measurements = True
use_coloc_measurements       = True

spots_min_prominence         = 350.0 # MeasureSpotsOperator.get_default_prominence()

### B. Mesures dans les noyaux